# FOXF1_BMP4_timelapse — 04_review_preliminary_tracks

**Feeds:** Fig 5i, ED Fig 10f

**Position in the chain:** run the numbered notebooks in order

Ported unchanged from the original analysis: outputs are as they ran, and no code
cell was edited. Paths appear as `<analysis-root>/...`.


# FOXF1/BMP4 Timelapse: Preliminary Track Review

This notebook reviews the preliminary tracking handoff produced from the ilastik-driven segmentation.

The goal here is not to optimize the tracker further. The goal is to **inspect what we now have**:

1. summary QC and file paths
2. saved BF/RFP overlay montages
3. frame-by-frame track overlays at selected times
4. representative long tracks
5. suspicious / low-confidence links
6. confirmed division examples

This notebook should be fast because it reuses exported artifacts instead of recomputing segmentation.


In [ ]:
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile

from IPython.display import Image, display, Markdown
from skimage import segmentation

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 200)
np.set_printoptions(suppress=True, precision=4)


In [ ]:
# ----------------------------- #
# Configuration
# ----------------------------- #
def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for cand in [start] + list(start.parents):
        if (cand / "ilastik" / "datasets" / "20260213" / "batch_processing_data_v0").exists() and (cand / "scripts").exists():
            return cand
    return start


def find_latest_handoff_dir(root: Path) -> Path:
    base = root / "results" / "datasets" / "20260213" / "segmentation_03c_ilastik"
    candidates = sorted(base.glob("fiji_preliminary_tracks_*"), key=lambda p: p.stat().st_mtime)
    if not candidates:
        raise FileNotFoundError(f"No handoff directories found under {base}")
    return candidates[-1]


ROOT = find_project_root(Path.cwd())
RAW_PATH = ROOT / "ilastik" / "datasets" / "20260213" / "batch_processing_data_v0" / "1-Pos009_012.tif"
HANDOFF_DIR = find_latest_handoff_dir(ROOT)
POSITION_LABEL = "1-Pos009_012"

DET_PATH = HANDOFF_DIR / "detections_tracks.csv"
EDGE_PATH = HANDOFF_DIR / "track_edges.csv"
TRACK_PATH = HANDOFF_DIR / "track_summary.csv"
DIV_PATH = HANDOFF_DIR / "division_events.csv"
QC_PATH = HANDOFF_DIR / "tracking_qc_summary.csv"
LABELS_PATH = HANDOFF_DIR / "tracked_label_stack_uint16.tif"
STATUS_PATH = HANDOFF_DIR / "trackmate_export_status.json"
TRACK_MOVIE_DIR = HANDOFF_DIR / "track_qc_movies"
HQ_TRACKS_PATH = TRACK_MOVIE_DIR / "high_quality_track_qc.csv"
HQ_TRACK_OUTPUTS_PATH = TRACK_MOVIE_DIR / "high_quality_track_qc_outputs.csv"

FRAME_PREVIEW_TIMES = [0, 30, 75, 120, 150, 225, 299]
N_LONG_TRACKS_TO_SHOW = 6
N_LOW_CONF_LINKS_TO_SHOW = 8
N_DIVISIONS_TO_SHOW = 6
TRACK_CROP_HALF_SIZE = 36
LINK_CROP_HALF_SIZE = 36
DIV_CROP_HALF_SIZE = 42

print("ROOT:", ROOT)
print("HANDOFF_DIR:", HANDOFF_DIR)


## 1) Load Handoff Artifacts


In [ ]:
assert RAW_PATH.exists(), f"Missing raw image stack: {RAW_PATH}"
for p in [DET_PATH, EDGE_PATH, TRACK_PATH, DIV_PATH, QC_PATH, LABELS_PATH, STATUS_PATH]:
    assert p.exists(), f"Missing artifact: {p}"

det_df = pd.read_csv(DET_PATH)
edge_df = pd.read_csv(EDGE_PATH)
track_df = pd.read_csv(TRACK_PATH)
div_df = pd.read_csv(DIV_PATH)
qc_df = pd.read_csv(QC_PATH)
with open(STATUS_PATH) as f:
    trackmate_status = json.load(f)

raw_mm = tifffile.memmap(RAW_PATH)
tracked_labels = tifffile.memmap(LABELS_PATH)
hq_tracks_df = pd.read_csv(HQ_TRACKS_PATH) if HQ_TRACKS_PATH.exists() else pd.DataFrame()
hq_track_outputs_df = pd.read_csv(HQ_TRACK_OUTPUTS_PATH) if HQ_TRACK_OUTPUTS_PATH.exists() else pd.DataFrame()

assert raw_mm.ndim == 4 and raw_mm.shape[1] == 2, f"Unexpected raw shape: {raw_mm.shape}"
assert tracked_labels.ndim == 3, f"Unexpected label shape: {tracked_labels.shape}"
assert raw_mm.shape[0] == tracked_labels.shape[0], "Time dimension mismatch between raw images and tracked labels."

N_TIME = int(raw_mm.shape[0])
print("raw shape:", raw_mm.shape, "dtype:", raw_mm.dtype)
print("tracked label shape:", tracked_labels.shape, "dtype:", tracked_labels.dtype)
print("detections:", len(det_df), "tracks:", len(track_df), "edges:", len(edge_df), "divisions:", len(div_df))


## 2) Summary Metrics And Key Files


In [ ]:
display(qc_df)

summary_table = pd.DataFrame(
    [
        {"artifact": "TrackMate session", "path": str(HANDOFF_DIR / "trackmate_session.xml")},
        {"artifact": "Linear TrackMate XML fallback", "path": str(HANDOFF_DIR / "trackmate_linear_tracks.xml")},
        {"artifact": "Tracked label stack", "path": str(LABELS_PATH)},
        {"artifact": "Detection table", "path": str(DET_PATH)},
        {"artifact": "Edge table", "path": str(EDGE_PATH)},
        {"artifact": "Track summary", "path": str(TRACK_PATH)},
        {"artifact": "Division events", "path": str(DIV_PATH)},
    ]
)
display(summary_table)

print("TrackMate export status:")
print(json.dumps(trackmate_status, indent=2))


In [ ]:
display(Markdown("### Track length summary"))
display(track_df["length"].describe().to_frame().T)

display(Markdown("### Edge type counts"))
display(edge_df["edge_type"].value_counts(dropna=False).rename_axis("edge_type").to_frame("count"))

display(Markdown("### Top 15 longest tracks"))
top_long_table = (
    track_df.sort_values(["length", "mean_edge_confidence"], ascending=[False, False])
    .head(15)
    .reset_index(drop=True)
)
display(top_long_table)

if not hq_tracks_df.empty:
    display(Markdown("### Curated high-quality track movies"))
    display(hq_tracks_df)
    display(Markdown("### High-quality track movie files"))
    display(hq_track_outputs_df)

display(Markdown("### First confirmed divisions"))
display(div_df.head(15))


## 3) Saved Overview Outputs

These are the lighter pre-rendered overview figures from the handoff pipeline.
The full montage PNGs remain on disk in the handoff folder, but they are intentionally
not embedded here because they make the executed notebook extremely large.


In [ ]:
overview_files = [
    HANDOFF_DIR / "sample_track_overlays.png",
    HANDOFF_DIR / "tracking_qc_plots.png",
]

for p in overview_files:
    display(Markdown(f"### `{p.name}`"))
    display(Image(filename=str(p)))

display(Markdown("### Large montage files kept on disk"))
display(
    pd.DataFrame(
        [
            {"artifact": "BF montage PNG", "path": str(HANDOFF_DIR / "tracked_overlay_montage_bf.png")},
            {"artifact": "RFP montage PNG", "path": str(HANDOFF_DIR / "tracked_overlay_montage_rfp.png")},
            {"artifact": "BF overlay stack TIFF", "path": str(HANDOFF_DIR / "tracked_overlay_stack_bf.tif")},
            {"artifact": "RFP overlay stack TIFF", "path": str(HANDOFF_DIR / "tracked_overlay_stack_rfp.tif")},
        ]
    )
)


## 4) Helpers For Interactive Inspection


In [ ]:
det_by_id = det_df.set_index("det_id")


def get_frame_images(t: int):
    bf = raw_mm[int(t), 0].astype(np.float32)
    rfp = raw_mm[int(t), 1].astype(np.float32)
    labels = tracked_labels[int(t)].astype(np.int32)
    return bf, rfp, labels


def robust_rescale(gray: np.ndarray, valid_mask: np.ndarray | None = None, q_low: float = 1.0, q_high: float = 99.5):
    if valid_mask is None:
        valid_mask = np.ones_like(gray, dtype=bool)
    vals = gray[valid_mask]
    if vals.size:
        lo = float(np.percentile(vals, q_low))
        hi = float(np.percentile(vals, q_high))
    else:
        lo = float(np.min(gray))
        hi = float(np.max(gray))
    if hi <= lo:
        hi = lo + 1.0
    out = np.clip((gray.astype(np.float32) - lo) / (hi - lo), 0.0, 1.0)
    out = out.copy()
    out[~valid_mask] = 0.0
    return out


def overlay_single_track(gray: np.ndarray, labels: np.ndarray, track_id: int):
    base = robust_rescale(gray)
    rgb = np.dstack([base, base, base])
    mask = labels == int(track_id)
    if np.any(mask):
        boundary = segmentation.find_boundaries(mask, mode="outer")
        rgb[..., 0][boundary] = 1.0
        rgb[..., 1][boundary] = 0.2
        rgb[..., 2][boundary] = 0.2
    return rgb


def overlay_all_tracks(gray: np.ndarray, labels: np.ndarray):
    base = robust_rescale(gray)
    rgb = np.dstack([base, base, base])
    mask = labels > 0
    return segmentation.mark_boundaries(rgb, mask, color=(1.0, 0.2, 0.2), mode="thick")


def crop_bounds(cy: float, cx: float, half_size: int, shape: tuple[int, int]):
    h, w = shape
    y0 = max(0, int(round(cy)) - half_size)
    y1 = min(h, int(round(cy)) + half_size)
    x0 = max(0, int(round(cx)) - half_size)
    x1 = min(w, int(round(cx)) + half_size)
    return y0, y1, x0, x1


def plot_frame_preview(times: list[int]):
    times = [int(t) for t in times if 0 <= int(t) < N_TIME]
    fig, axes = plt.subplots(len(times), 2, figsize=(8, 3.2 * len(times)), constrained_layout=True)
    if len(times) == 1:
        axes = np.array([axes])
    for r, t in enumerate(times):
        bf, rfp, labels = get_frame_images(t)
        axes[r, 0].imshow(overlay_all_tracks(bf, labels))
        axes[r, 0].set_title(f"BF overlay t={t} | n={int((labels > 0).max() and labels.max())}")
        axes[r, 1].imshow(overlay_all_tracks(rfp, labels))
        axes[r, 1].set_title(f"RFP overlay t={t}")
        for c in range(2):
            axes[r, c].set_xticks([])
            axes[r, c].set_yticks([])
    plt.show()


def plot_track_crops(track_id: int, n_points: int = 5, half_size: int = TRACK_CROP_HALF_SIZE):
    g = det_df.loc[det_df["track_id"] == int(track_id)].sort_values("time").reset_index(drop=True)
    if g.empty:
        print(f"Track {track_id} not found.")
        return
    idx = np.unique(np.linspace(0, len(g) - 1, min(n_points, len(g)), dtype=int))
    sel = g.iloc[idx].reset_index(drop=True)
    fig, axes = plt.subplots(len(sel), 2, figsize=(8, 3 * len(sel)), constrained_layout=True)
    if len(sel) == 1:
        axes = np.array([axes])
    for r, row in enumerate(sel.itertuples(index=False)):
        bf, rfp, labels = get_frame_images(int(row.time))
        y0, y1, x0, x1 = crop_bounds(float(row.cy), float(row.cx), half_size, bf.shape)
        axes[r, 0].imshow(overlay_single_track(bf, labels, int(track_id))[y0:y1, x0:x1])
        axes[r, 0].set_title(f"BF track {track_id} | t={int(row.time)} | idx={int(row.track_index)+1}/{int(row.track_len)}")
        axes[r, 1].imshow(overlay_single_track(rfp, labels, int(track_id))[y0:y1, x0:x1])
        axes[r, 1].set_title(f"RFP | area={float(row.area):.0f} | prob={float(row.mean_prob):.3f}")
        for c in range(2):
            axes[r, c].set_xticks([])
            axes[r, c].set_yticks([])
    plt.show()


def plot_link_examples(link_rows: pd.DataFrame, half_size: int = LINK_CROP_HALF_SIZE):
    if link_rows.empty:
        print("No links to show.")
        return
    n = len(link_rows)
    fig, axes = plt.subplots(n, 2, figsize=(8, 3 * n), constrained_layout=True)
    if n == 1:
        axes = np.array([axes])
    for r, row in enumerate(link_rows.itertuples(index=False)):
        src = det_by_id.loc[int(row.source_det_id)]
        dst = det_by_id.loc[int(row.target_det_id)]
        for c, (det_row, title_prefix) in enumerate([(src, "source"), (dst, "target")]):
            bf, rfp, labels = get_frame_images(int(det_row["time"]))
            y0, y1, x0, x1 = crop_bounds(float(det_row["cy"]), float(det_row["cx"]), half_size, bf.shape)
            img = overlay_single_track(rfp, labels, int(det_row["track_id"]))
            axes[r, c].imshow(img[y0:y1, x0:x1])
            axes[r, c].set_title(
                f"{title_prefix} t={int(det_row['time'])} | track={int(det_row['track_id'])}\n"
                f"{row.edge_type} | conf={float(row.confidence):.3f} | gap={int(row.gap_missing)}"
            )
            axes[r, c].set_xticks([])
            axes[r, c].set_yticks([])
    plt.show()


def plot_division_examples(division_rows: pd.DataFrame, half_size: int = DIV_CROP_HALF_SIZE):
    if division_rows.empty:
        print("No confirmed divisions to show.")
        return
    n = len(division_rows)
    fig, axes = plt.subplots(n, 3, figsize=(12, 3.2 * n), constrained_layout=True)
    if n == 1:
        axes = np.array([axes])
    for r, row in enumerate(division_rows.itertuples(index=False)):
        parent_track = int(row.parent_track_id)
        d1 = int(row.daughter_track_id_1)
        d2 = int(row.daughter_track_id_2)
        parent_row = (
            det_df.loc[det_df["track_id"] == parent_track].sort_values("time").iloc[-1]
        )
        d1_row = det_df.loc[det_df["track_id"] == d1].sort_values("time").iloc[0]
        d2_row = det_df.loc[det_df["track_id"] == d2].sort_values("time").iloc[0]

        parent_frame = int(parent_row["time"])
        daughter_frame = int(d1_row["time"])

        bf_p, rfp_p, lab_p = get_frame_images(parent_frame)
        bf_d, rfp_d, lab_d = get_frame_images(daughter_frame)

        cy = np.mean([float(parent_row["cy"]), float(d1_row["cy"]), float(d2_row["cy"])])
        cx = np.mean([float(parent_row["cx"]), float(d1_row["cx"]), float(d2_row["cx"])])
        y0, y1, x0, x1 = crop_bounds(cy, cx, half_size, bf_p.shape)

        axes[r, 0].imshow(overlay_single_track(rfp_p, lab_p, parent_track)[y0:y1, x0:x1])
        axes[r, 0].set_title(f"Parent track {parent_track}\nframe={parent_frame}")

        mix = overlay_single_track(rfp_d, lab_d, d1)
        mask_d2 = lab_d == d2
        if np.any(mask_d2):
            boundary_d2 = segmentation.find_boundaries(mask_d2, mode='outer')
            mix[..., 0][boundary_d2] = 0.2
            mix[..., 1][boundary_d2] = 1.0
            mix[..., 2][boundary_d2] = 0.2
        axes[r, 1].imshow(mix[y0:y1, x0:x1])
        axes[r, 1].set_title(f"Daughters frame={daughter_frame}\nred={d1}, green={d2}")

        axes[r, 2].imshow(overlay_all_tracks(bf_d, lab_d)[y0:y1, x0:x1])
        axes[r, 2].set_title("BF context")

        for c in range(3):
            axes[r, c].set_xticks([])
            axes[r, c].set_yticks([])
    plt.show()


## 5) Selected Frame Previews

These are useful for quickly checking whether the preliminary tracks look sensible at sparse and crowded timepoints.


In [ ]:
plot_frame_preview(FRAME_PREVIEW_TIMES)


## 6) Representative Long Tracks


In [ ]:
long_tracks = (
    track_df.sort_values(["length", "mean_edge_confidence"], ascending=[False, False])
    .head(N_LONG_TRACKS_TO_SHOW)
    .reset_index(drop=True)
)
display(long_tracks)


In [ ]:
for row in long_tracks.itertuples(index=False):
    display(Markdown(f"### Track `{int(row.track_id)}` | length={int(row.length)} | mean edge confidence={float(row.mean_edge_confidence):.3f}"))
    plot_track_crops(int(row.track_id))


## 7) Curated High-Quality Track Movies

These are the new temporal QC artifacts: each selected track has a cropped TIFF stack
plus a small preview montage sampled across its lifetime. This is a better way to judge
track quality than isolated single-frame snapshots.


In [ ]:
if hq_track_outputs_df.empty:
    print("No curated track movies found in the handoff folder.")
else:
    for row in hq_track_outputs_df.itertuples(index=False):
        display(
            Markdown(
                f"### Track `{int(row.track_id)}` | length={int(row.length)} | "
                f"mean edge confidence={float(row.mean_edge_confidence):.3f}\n"
                f"- stack: `{row.overlay_stack_tif}`"
            )
        )
        p = Path(row.preview_montage_png)
        if p.exists():
            display(Image(filename=str(p)))


## 8) Low-Confidence Links To Inspect

These are not guaranteed failures, but they are the places where the tracker itself was least certain.


In [ ]:
low_conf_links = (
    edge_df.sort_values(["confidence", "cost"], ascending=[True, False])
    .head(N_LOW_CONF_LINKS_TO_SHOW)
    .reset_index(drop=True)
)
display(low_conf_links)


In [ ]:
plot_link_examples(low_conf_links)


## 9) Confirmed Division Examples


In [ ]:
div_show = div_df.head(N_DIVISIONS_TO_SHOW).reset_index(drop=True)
display(div_show)
plot_division_examples(div_show)


## 10) Manual Inspection Helper

Change the values below and rerun this cell when you want to inspect a specific track.


In [ ]:
TRACK_ID_TO_INSPECT = int(long_tracks.iloc[0]["track_id"]) if len(long_tracks) else 1
plot_track_crops(TRACK_ID_TO_INSPECT, n_points=6)
